# Understanding BDH — from scratch

**What you need to know:** basic deep learning — layers, weights, ReLU, loss. That's it.  
No Transformer knowledge required.

### What is BDH?

BDH is a **language model** — it reads text and predicts what comes next, byte by byte.  
What makes it unusual compared to a standard neural net:

| | Standard dense MLP | BDH |
|---|---|---|
| Activations | All neurons fire (dense) | ~5% of neurons fire (sparse) |
| Processes sequences? | No — each input independent | Yes — tokens look at past tokens |
| Memory with context? | N/A | Fixed-size state (doesn't grow) |
| Works on raw text? | Needs tokenizer | Works on raw bytes directly |

### What you'll build

By the end of this notebook you'll have a working language model trained on Shakespeare — and you'll understand every line of how it works.

---
## Part 1 — PyTorch basics

### Tensors are just arrays with shapes

A tensor is a grid of numbers. The **shape** tells you its dimensions.

In [ ]:
import torch

a = torch.tensor([1.0, 2.0, 3.0])   # 1D — a list of 3 numbers
b = torch.tensor([[1.0, 2.0],        # 2D — a 2x3 grid (matrix)
                  [3.0, 4.0],
                  [5.0, 6.0]])

print(a.shape)   # torch.Size([3])
print(b.shape)   # torch.Size([3, 2])  ← 3 rows, 2 columns

### The `@` operator = matrix multiply

This is the most important operation in all of deep learning. When you multiply two matrices, the inner dimensions must match.

In [ ]:
# [3, 2] @ [2, 4]  →  [3, 4]
#  inner dims match (both 2) ↑↑, outer dims become the output shape

W = torch.randn(2, 4)   # a "weight matrix"
x = torch.randn(3, 2)   # some input

out = x @ W
print(f"x shape:   {x.shape}")
print(f"W shape:   {W.shape}")
print(f"out shape: {out.shape}")   # [3, 4]

### `ReLU` — the simplest activation

ReLU just kills negative numbers. That's it. This is how BDH gets **sparse** activations — most pre-activations are negative, so most neurons end up as zero.

In [ ]:
import torch.nn.functional as F

x = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])
print("before ReLU:", x)
print("after ReLU: ", F.relu(x))

# On a random vector, about 50% will be negative → 50% become zero
random = torch.randn(1000)
active = (F.relu(random) > 0).float().mean()
print(f"\nFraction active on random input: {active:.0%}")
print("(after training, BDH pushes this down to ~5%)")

### `nn.Module` — how every PyTorch model is built

Every model (and every layer inside it) is a class that inherits from `nn.Module`. You only need to write two methods:
- `__init__` — define your weights
- `forward` — describe what happens to an input

`nn.Parameter` wraps a tensor so the optimizer knows to update it.

In [ ]:
from torch import nn

class TinyLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        # nn.Parameter = a weight the optimizer will update
        self.W = nn.Parameter(torch.randn(in_dim, out_dim) * 0.02)

    def forward(self, x):
        return F.relu(x @ self.W)   # expand then sparsify

layer = TinyLayer(4, 8)
x = torch.randn(2, 4)          # batch of 2 inputs, each 4-dim
out = layer(x)                 # calls forward() automatically
print(f"input:  {x.shape}")
print(f"output: {out.shape}")
print(f"active: {(out > 0).float().mean():.0%}")

### Batches — processing many sequences at once

In practice we never process one input at a time. We stack `B` sequences into a batch so the GPU can work on them in parallel. This adds a dimension to the front.

The shape convention used throughout BDH:
```
B = batch size      (how many sequences at once)
T = sequence length (how many tokens)  
D = embedding dim   (numbers per token)
```

In [ ]:
B, T, D = 4, 16, 8   # 4 sequences, 16 tokens each, 8 numbers per token

x = torch.randn(B, T, D)
print(f"x.shape = {x.shape}  ← [B, T, D]")

# @ works on batched tensors too — it only touches the last two dims
W = torch.randn(D, 32)
out = x @ W
print(f"(x @ W).shape = {out.shape}  ← [B, T, 32]  (B and T are untouched)")

---
## Part 2 — The BDH model

### Connecting to what you know: from dense to sparse

In a standard MLP with N=512 hidden neurons, **all 512 fire** for every input:
```
standard layer:  [ 0.3, -0.1,  0.7,  0.2, -0.5, ... ]   ← all neurons active
```
BDH instead uses N=8192 neurons but forces ~95% to zero with ReLU:
```
BDH layer:       [ 0.0,  0.0,  0.4,  0.0,  0.0, ... ]   ← only ~5% fire
```

Why? In a large enough space, each neuron can specialize in a single concept without interfering with others. A neuron that fires for "past-tense verb" simply won't fire for "noun". This specialization is called **monosemanticity** and makes the model interpretable.

BDH also adds **attention** — a mechanism letting each token look at past tokens and gather relevant information before computing its sparse pattern.

The architecture diagram below shows the full picture.

### Step 1: Config

All hyperparameters in one dataclass.  
Key values: `n_embd` (D — compressed dimension between layers), `mlp_internal_dim_multiplier` (how much to expand to neurons), `n_layer` (same weights applied this many times).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(11, 9))
ax.set_xlim(0, 11); ax.set_ylim(0, 10); ax.axis('off')
fig.patch.set_facecolor('#FAFAFA'); ax.set_facecolor('#FAFAFA')

def rb(cx, cy, w, h, title, sub='', fc='#1565C0', tc='white', fs=11):
    ax.add_patch(FancyBboxPatch((cx-w/2, cy-h/2), w, h, boxstyle='round,pad=0.12',
                  facecolor=fc, edgecolor='white', linewidth=2.5, zorder=3))
    dy = 0.18 if sub else 0
    ax.text(cx, cy+dy, title, ha='center', va='center', fontsize=fs,
            fontweight='bold', color=tc, zorder=4)
    if sub:
        ax.text(cx, cy-0.22, sub, ha='center', va='center', fontsize=8.5, color=tc, alpha=0.9, zorder=4)

def arr(cx, y0, y1, lbl=''):
    ax.annotate('', xy=(cx, y1+0.04), xytext=(cx, y0-0.04),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2.2), zorder=5)
    if lbl:
        ax.text(cx+0.2, (y0+y1)/2, lbl, ha='left', va='center', fontsize=8.5, color='#777', zorder=5)

# Input
ax.text(5.5, 9.55, '"hello"', ha='center', fontsize=14, color='#333', family='monospace',
        bbox=dict(boxstyle='round,pad=0.4', fc='#FFF9C4', ec='#F9A825', lw=1.5))
ax.text(5.5, 9.05, 'encoded as bytes: [104, 101, 108, 108, 111]   (no tokenizer needed)',
        ha='center', fontsize=9.5, color='#555', style='italic')

arr(5.5, 8.8, 8.38, '[B, T]  integer indices')

rb(5.5, 8.08, 7, 0.65, 'Token Embedding',
   'each of 256 possible bytes  →  a learnable vector of D=256 numbers', fc='#1976D2')

arr(5.5, 7.75, 7.3, '[B, T, D=256]')

# Layer group background
ax.add_patch(FancyBboxPatch((1.3, 3.65), 8.4, 3.45, boxstyle='round,pad=0.2',
             facecolor='#E3F2FD', edgecolor='#1976D2', linewidth=2.5, linestyle='--', zorder=2))
ax.text(5.5, 7.0, 'BDH Layer  ×  n_layer   (same weights reused every iteration)',
        ha='center', fontsize=10.5, fontweight='bold', color='#0D47A1')

# 4 steps inside the layer
c_list = ['#1565C0', '#00695C', '#2E7D32', '#6A1B9A']
steps  = [('① Expand',  'D=256 → N=8192\nper head'),
          ('② ReLU\n(sparse!)', '~95% neurons\n→ exactly 0'),
          ('③ Linear\nAttention',  'tokens look at\npast tokens (O(T))'),
          ('④ Hebbian\nGate  ×',   'x_sparse × y_sparse\n"fire together"')]
xs = [2.5, 4.3, 6.8, 9.2]
for (t, s), x, fc in zip(steps, xs, c_list):
    rb(x, 5.85, 1.65, 1.5, t, s, fc=fc, fs=9.5)

for i in range(len(xs)-1):
    ax.annotate('', xy=(xs[i+1]-0.83, 5.85), xytext=(xs[i]+0.83, 5.85),
                arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2.0), zorder=5)

ax.text(5.5, 4.3,
        '⑤  compress back:  N × heads  →  D=256    then   output + input  ( residual )',
        ha='center', fontsize=9.5, color='#333',
        bbox=dict(boxstyle='round,pad=0.35', fc='#FFF9C4', ec='#F9A825', lw=1.5, zorder=4))

arr(5.5, 3.65, 3.22, '[B, T, D=256]')

rb(5.5, 2.92, 7, 0.65, 'Output Projection',
   'D=256  →  256 logits  (one score per possible next byte)', fc='#2E7D32')

arr(5.5, 2.59, 2.16, 'softmax → probabilities')

rb(5.5, 1.86, 5.5, 0.55, 'Sample next byte  →  append  →  repeat', '', fc='#BF360C')

ax.set_title('BDH — Architecture Overview', fontsize=15, fontweight='bold', color='#111', pad=10)
plt.tight_layout()
plt.show()

In [ ]:
import dataclasses, math

@dataclasses.dataclass
class BDHConfig:
    n_layer:                    int   = 6    # how many times to loop through the weights
    n_embd:                     int   = 256  # D — small "communication" dimension
    dropout:                    float = 0.1
    n_head:                     int   = 4    # parallel processing heads
    mlp_internal_dim_multiplier: int  = 128  # controls size of neuron space
    vocab_size:                 int   = 256  # byte-level (0-255)

cfg = BDHConfig()
D  = cfg.n_embd
nh = cfg.n_head
N  = cfg.mlp_internal_dim_multiplier * D // nh

print(f"D (embedding dim)  = {D}")
print(f"nh (heads)         = {nh}")
print(f"N (neurons/head)   = {N:,}")
print(f"Total neuron space = {N * nh:,}  ← huge sparse space BDH operates in")

### Step 2: Attention — how tokens look at each other

In a plain MLP, every input is processed in isolation. Attention lets each token **look back at previous tokens**.

**The core question attention answers:**  
*"Given what I am (current token), which past tokens are similar to me, and what were they representing?"*

**BDH's version — simpler than Transformer attention:**
- Uses the **same sparse pattern** for Q (query) and K (key) — no separate projection matrices
- Uses **raw dot products** (no softmax) — weights can be any value, not just 0-to-1
- The causal mask (`.tril(diagonal=-1)`) ensures each token only sees **past** tokens, not itself or future ones
- At inference, this sum can be accumulated as a **fixed-size matrix** `ρ = Σ (v ⊗ k)` — memory stays constant no matter how long the sequence

**RoPE (Rotary Position Embedding):** Rotates Q and K vectors by their position index before the dot product. Tokens far apart get rotated more relative to each other, naturally reducing their dot product. This is how the model tracks word order without adding position vectors. Ignore the implementation details for now.

The diagram below compares Transformer attention to BDH's linear attention.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7.5))
fig.patch.set_facecolor('#FAFAFA')
fig.suptitle('Attention Comparison: Transformer vs BDH', fontsize=14, fontweight='bold', y=0.98)

for ax in (ax1, ax2):
    ax.set_xlim(0, 8); ax.set_ylim(0, 10); ax.axis('off'); ax.set_facecolor('#FAFAFA')

def rb(ax, cx, cy, w, h, t, s='', fc='#1565C0', tc='white', fs=10):
    ax.add_patch(FancyBboxPatch((cx-w/2, cy-h/2), w, h, boxstyle='round,pad=0.12',
                  facecolor=fc, edgecolor='white', linewidth=2, zorder=3))
    dy = 0.18 if s else 0
    ax.text(cx, cy+dy, t, ha='center', va='center', fontsize=fs, fontweight='bold', color=tc, zorder=4)
    if s:
        ax.text(cx, cy-0.23, s, ha='center', va='center', fontsize=8, color=tc, alpha=0.88, zorder=4)

def av(ax, cx, y0, y1, c='#555'):
    ax.annotate('', xy=(cx, y1+0.04), xytext=(cx, y0-0.04),
                arrowprops=dict(arrowstyle='->', color=c, lw=1.8), zorder=5)

# ── LEFT: Transformer ──────────────────────────────────────────────────────────
ax1.set_title('Standard Transformer Attention', fontsize=12, fontweight='bold', color='#B71C1C', pad=6)
rb(ax1, 4, 9.3, 5.5, 0.55, 'Input  x  [T, D]', '', fc='#455A64')
av(ax1, 4, 9.0, 8.5)
ax1.text(4, 8.25, '3 separate learned projections (Wq, Wk, Wv)', ha='center', fontsize=9, color='#B71C1C')
for lbl, x in [('Q', 1.8), ('K', 4.0), ('V', 6.2)]:
    rb(ax1, x, 7.8, 1.7, 0.55, lbl, '', fc='#C62828', fs=11)
    ax1.annotate('', xy=(x, 8.0), xytext=(4, 8.5),
                 arrowprops=dict(arrowstyle='->', color='#B71C1C', lw=1.3), zorder=5)

rb(ax1, 4, 6.7, 5.5, 0.6, 'scores = Q @ Kᵀ  /  √d',
   '[T, T] matrix — every token vs every other', fc='#E53935')
av(ax1, 4, 7.5, 7.05)
rb(ax1, 4, 5.7, 5.5, 0.6, 'weights = softmax(scores)',
   'rows sum to 1.0 — purely relative', fc='#D32F2F')
av(ax1, 4, 6.4, 6.05)
rb(ax1, 4, 4.7, 5.5, 0.6, 'output = weights @ V',
   'weighted average of value vectors', fc='#C62828')
av(ax1, 4, 5.4, 5.05)
av(ax1, 4, 4.4, 3.95)
rb(ax1, 4, 3.65, 5.5, 0.55, 'Output  [T, D]', '', fc='#455A64')

ax1.add_patch(FancyBboxPatch((0.4, 0.2), 7.2, 3.15, boxstyle='round,pad=0.1',
              facecolor='#FFEBEE', edgecolor='#EF9A9A', lw=1.5, zorder=2))
ax1.text(4, 3.1, 'Limitations', ha='center', fontsize=10, fontweight='bold', color='#B71C1C')
for y, t in [(2.7, '✗  O(T²) memory — KV-cache grows with context'),
             (2.3, '✗  Softmax normalizes: hard to be truly sparse'),
             (1.9, '✗  3 weight matrices per layer (Wq, Wk, Wv)'),
             (1.5, '✗  Cannot naturally do Hebbian updates'),
             (1.1, '✗  Weights sum to 1 — purely relative comparison')]:
    ax1.text(0.7, y, t, fontsize=9, color='#C62828')

# ── RIGHT: BDH ─────────────────────────────────────────────────────────────────
ax2.set_title('BDH Linear Attention', fontsize=12, fontweight='bold', color='#1B5E20', pad=6)
rb(ax2, 4, 9.3, 5.5, 0.55, 'Input  x  [B, 1, T, D]', '', fc='#455A64')
av(ax2, 4, 9.0, 8.55)
rb(ax2, 4, 8.25, 5.5, 0.6, 'x_sparse = ReLU(x @ encoder)  [B, nh, T, N]',
   'one matrix, no separate Q/K projections', fc='#1565C0')
av(ax2, 4, 7.95, 7.5)
ax2.text(4, 7.3, 'Q = K = x_sparse    (same tensor — no extra weight matrices!)',
         ha='center', fontsize=9, color='#0D47A1', fontweight='bold')
av(ax2, 4, 7.15, 6.75)
rb(ax2, 4, 6.45, 5.5, 0.6, 'scores = (Q @ Kᵀ).tril(diagonal=-1)',
   'no softmax, no scaling — raw dot products with causal mask', fc='#0277BD')
av(ax2, 4, 6.15, 5.7)
rb(ax2, 4, 5.4, 5.5, 0.6, 'a_star = scores @ V    (V = original x)',
   'at inference: ρ += v ⊗ k  — fixed-size memory!', fc='#01579B')
av(ax2, 4, 5.1, 4.65)
rb(ax2, 4, 4.35, 5.5, 0.6, 'y_sparse = ReLU(a_star @ encoder_v)',
   'expand attention output to neuron space', fc='#0277BD')
av(ax2, 4, 4.05, 3.6)
rb(ax2, 4, 3.3, 5.5, 0.6, 'gate = x_sparse × y_sparse',
   'Hebbian rule: only neurons active in BOTH survive', fc='#1565C0')
av(ax2, 4, 3.0, 2.55)
rb(ax2, 4, 2.25, 5.5, 0.55, 'Output  [B, 1, T, D]', '', fc='#455A64')

ax2.add_patch(FancyBboxPatch((0.4, 0.2), 7.2, 1.85, boxstyle='round,pad=0.1',
              facecolor='#E8F5E9', edgecolor='#A5D6A7', lw=1.5, zorder=2))
ax2.text(4, 1.8, 'Advantages', ha='center', fontsize=10, fontweight='bold', color='#1B5E20')
for y, t in [(1.4, '✓  Fixed-size memory ρ — no KV-cache growth'),
             (1.0, '✓  ~5% sparsity → monosemantic, interpretable neurons'),
             (0.6, '✓  Q=K: fewer parameters, natural Hebbian update')]:
    ax2.text(0.7, y, t, fontsize=9, color='#1B5E20')

plt.tight_layout()
plt.show()

In [ ]:
def get_freqs(n, theta, dtype):
    def quantize(t, q=2):
        return (t / q).floor() * q
    return (1.0 / (theta ** (quantize(torch.arange(0, n, 1, dtype=dtype)) / n)) / (2 * math.pi))


class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        nh = config.n_head
        D  = config.n_embd
        N  = config.mlp_internal_dim_multiplier * D // nh

        # Pre-computed position frequencies — not a learned weight, just a lookup table
        # torch.nn.Buffer: moves to GPU with the model but the optimizer ignores it
        self.freqs = torch.nn.Buffer(
            get_freqs(N, theta=2**16, dtype=torch.float32).view(1, 1, 1, N)
        )

    @staticmethod
    def rope(phases, v):
        """Rotate vector v by the given phase angles (positional encoding)."""
        phases  = (phases % 1) * (2 * math.pi)
        v_rot   = torch.stack((-v[..., 1::2], v[..., ::2]), dim=-1).view(*v.size())
        return (v * torch.cos(phases)).to(v.dtype) + (v_rot * torch.sin(phases)).to(v.dtype)

    def forward(self, Q, K, V):
        assert K is Q   # in BDH, queries and keys are the exact same tensor
        _, _, T, _ = Q.size()

        # Build rotation angles: position 0 gets angle 0, position 1 gets freq, etc.
        r_phases = torch.arange(0, T, device=self.freqs.device, dtype=self.freqs.dtype).view(1,1,-1,1) * self.freqs

        QR = self.rope(r_phases, Q)   # rotate Q by position
        KR = QR                       # K is Q, so same rotation

        # scores[t, s] = "how much should token t attend to token s?"
        # .tril(diagonal=-1) zeros out s >= t so tokens only see the PAST
        scores = (QR @ KR.mT).tril(diagonal=-1)   # [B, nh, T, T]

        # No softmax — raw scores. Weighted sum of V.
        return scores @ V   # [B, nh, T, D]

print("Attention defined ✓")

### Step 3: The full BDH model

Here's the core loop — same weights reused `n_layer` times:

```
x  →  expand to neuron space  →  ReLU (sparse!)
                                      ↓
                              linear attention
                                      ↓
                  expand attention output  →  ReLU (sparse!)
                                      ↓
              x_sparse  ×  y_sparse  =  Hebbian gate
                                      ↓
                         compress back to D  →  residual  →  repeat
```

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(14, 6.5))
ax.set_xlim(0, 14); ax.set_ylim(0, 7); ax.axis('off')
fig.patch.set_facecolor('#FAFAFA'); ax.set_facecolor('#FAFAFA')

def rb(cx, cy, w, h, t, s='', fc='#1565C0', tc='white', fs=10.5):
    ax.add_patch(FancyBboxPatch((cx-w/2, cy-h/2), w, h, boxstyle='round,pad=0.12',
                  facecolor=fc, edgecolor='white', linewidth=2, zorder=3))
    dy = 0.18 if s else 0
    ax.text(cx, cy+dy, t, ha='center', va='center', fontsize=fs, fontweight='bold', color=tc, zorder=4)
    if s:
        ax.text(cx, cy-0.22, s, ha='center', va='center', fontsize=8, color=tc, alpha=0.9, zorder=4)

def ah(x0, x1, cy, c='#1565C0', lbl=''):
    ax.annotate('', xy=(x1-0.05, cy), xytext=(x0+0.05, cy),
                arrowprops=dict(arrowstyle='->', color=c, lw=2), zorder=5)
    if lbl:
        ax.text((x0+x1)/2, cy+0.22, lbl, ha='center', va='bottom', fontsize=8, color=c, zorder=5)

def av(cx, y0, y1, c='#1565C0'):
    ax.annotate('', xy=(cx, y1+0.05), xytext=(cx, y0-0.05),
                arrowprops=dict(arrowstyle='->', color=c, lw=1.8), zorder=5)

# Input
rb(1.0, 5.5, 1.7, 0.65, 'v*', '[B,1,T,D]\nfrom prev layer', fc='#455A64', fs=10)
ah(1.85, 3.05, 5.5, '#1565C0', 'ln(v*)')

# Step 1: Expand
rb(4.1, 5.5, 2.0, 0.75, '① Expand', 'v* @ encoder\nD → N per head', fc='#1565C0')
ah(5.1, 6.3, 5.5, '#00695C', 'ReLU →')

# Step 2: x_sparse (Q = K)
rb(7.0, 5.5, 1.7, 0.75, '② x_sparse', '[B,nh,T,N]\n~50% nonzero*', fc='#00695C')
ax.text(7.0, 4.88, 'Q = K = x_sparse', ha='center', fontsize=8.5, color='#00695C', style='italic')

# V = original x (skip connection)
ax.plot([1.0, 1.0, 8.55], [5.17, 3.0, 3.0], color='#AAA', lw=1.8, ls='--', zorder=1)
ax.annotate('', xy=(8.55, 3.0), xytext=(8.5, 3.0),
            arrowprops=dict(arrowstyle='->', color='#AAA', lw=1.5), zorder=5)
ax.text(4.5, 2.78, 'V = x  (the original D-dim input, not expanded)', ha='center',
        fontsize=8.5, color='#888', style='italic')

ah(7.85, 9.3, 5.5, '#2E7D32', '(Q@Kᵀ).tril() @ V')

# Step 3: Linear attention output
rb(10.4, 5.5, 1.9, 0.75, '③ a_star', 'attention output\n[B,nh,T,D]', fc='#2E7D32')
ah(11.35, 12.55, 5.5, '#6A1B9A', 'ReLU(·@enc_v)')

# Step 4: y_sparse
rb(13.0, 5.5, 1.8, 0.75, '④ y_sparse', '[B,nh,T,N]\nfor gating', fc='#6A1B9A')

# Gate (down from x_sparse and y_sparse)
av(7.0, 5.12, 3.75, '#00695C')
av(13.0, 5.12, 3.75, '#6A1B9A')
rb(9.5, 3.4, 5.8, 0.65,
   '⑤  gate = x_sparse  ×  y_sparse   (element-wise)',
   'a neuron survives only if it fired for BOTH current token AND context', fc='#37474F')

# Compress
av(9.5, 3.07, 2.42, '#37474F')
rb(9.5, 2.08, 7.5, 0.65,
   '⑥  compress: gate.reshape → D   +   residual  (add original v*)',
   '', fc='#1565C0', fs=9.5)
# Residual skip
ax.plot([1.0, 1.0, 6.24], [5.17, 1.78, 1.78], color='#BBB', lw=1.8, ls=':', zorder=1)
ax.annotate('', xy=(6.24, 1.78), xytext=(6.2, 1.78),
            arrowprops=dict(arrowstyle='->', color='#BBB', lw=1.5), zorder=5)
ax.text(3.3, 1.55, 'residual +', ha='center', fontsize=8, color='#BBB', style='italic')

av(9.5, 1.75, 1.2, '#455A64')
rb(9.5, 0.88, 3.5, 0.55, 'v*_new  [B,1,T,D]', '', fc='#455A64')

ax.text(7.0, 4.58, '* drops to ~5% after training', ha='center', fontsize=7.5,
        color='#888', style='italic')

ax.set_title('Inside One BDH Layer — Data Flow', fontsize=14, fontweight='bold', color='#111', pad=8)
plt.tight_layout()
plt.show()

In [ ]:
class BDH(nn.Module):
    def __init__(self, config: BDHConfig):
        super().__init__()
        self.config = config
        nh = config.n_head
        D  = config.n_embd
        N  = config.mlp_internal_dim_multiplier * D // nh

        # The three shared weight matrices (reused every layer)
        self.encoder   = nn.Parameter(torch.zeros(nh, D, N).normal_(std=0.02))   # D → N
        self.encoder_v = nn.Parameter(torch.zeros(nh, D, N).normal_(std=0.02))   # D → N (for values)
        self.decoder   = nn.Parameter(torch.zeros(nh * N, D).normal_(std=0.02))  # nh*N → D

        self.attn  = Attention(config)
        self.ln    = nn.LayerNorm(D, elementwise_affine=False, bias=False)
        self.embed = nn.Embedding(config.vocab_size, D)
        self.drop  = nn.Dropout(config.dropout)
        self.lm_head = nn.Parameter(torch.zeros(D, config.vocab_size).normal_(std=0.02))

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        C  = self.config
        B, T = idx.size()
        D  = C.n_embd
        nh = C.n_head
        N  = D * C.mlp_internal_dim_multiplier // nh

        # Each token index (0-255) becomes a D-dim vector
        x = self.embed(idx).unsqueeze(1)   # [B, T, D] → [B, 1, T, D]  (add head dim)
        x = self.ln(x)

        for _ in range(C.n_layer):
            # 1. Expand: D → N per head, apply ReLU → sparse neuron activations
            x_sparse = F.relu(x @ self.encoder)          # [B, nh, T, N]

            # 2. Attention: each token asks "what did past tokens look like?"
            #    Q = K = x_sparse (same sparse activations used for both)
            #    V = x (the full D-dim representation)
            yKV = self.attn(Q=x_sparse, K=x_sparse, V=x)   # [B, nh, T, D]
            yKV = self.ln(yKV)

            # 3. Expand attention output to neuron space → sparse
            y_sparse = F.relu(yKV @ self.encoder_v)       # [B, nh, T, N]

            # 4. Hebbian gate: only neurons active in BOTH input AND context survive
            xy = self.drop(x_sparse * y_sparse)           # [B, nh, T, N]

            # 5. Compress back: nh*N → D, then residual
            yMLP = xy.transpose(1, 2).reshape(B, 1, T, N * nh) @ self.decoder  # [B, 1, T, D]
            x = self.ln(x + self.ln(yMLP))

        logits = x.view(B, T, D) @ self.lm_head   # [B, T, vocab_size]
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            idx = torch.cat((idx, torch.multinomial(F.softmax(logits, -1), 1)), dim=1)
        return idx

print("BDH defined ✓")

### Sanity check — does it even run?

In [ ]:
model = BDH(BDHConfig())

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")
print()

# Dummy batch: 2 sequences of 32 tokens (each token is a byte value 0-255)
dummy_input   = torch.randint(0, 256, (2, 32))
dummy_targets = torch.randint(0, 256, (2, 32))

logits, loss = model(dummy_input, dummy_targets)
print(f"Input shape:  {dummy_input.shape}")
print(f"Logits shape: {logits.shape}")       # [2, 32, 256]
print(f"Loss: {loss.item():.3f}")            # ~5.5 = log(256), random model
print()
print("(loss ~5.5 is expected — the model hasn't learned anything yet)")

---
## Part 2b — Inside the BDH forward pass, step by step

Let's trace through exactly what happens to one batch of data inside `BDH.forward()`.  
We'll do it manually, one operation at a time, so there's no mystery.

### Setup: a real model and a real input

We'll use the word "hello" encoded as raw bytes (BDH works at the byte level — no tokeniser needed).

In [ ]:
torch.manual_seed(0)
m = BDH(BDHConfig())
m.eval()

C  = m.config
D  = C.n_embd                                       # 256
nh = C.n_head                                       # 4
N  = C.mlp_internal_dim_multiplier * D // nh        # 8192

# Encode "hello" as bytes. Each character becomes one integer token (0-255).
text  = "hello"
idx   = torch.tensor([[ord(c) for c in text]])      # shape [1, 5]
B, T  = idx.shape

print(f"Input text: '{text}'")
print(f"As bytes:    {idx.tolist()[0]}")
print(f"idx shape:   {idx.shape}  ← [B=1, T=5]")
print(f"\nModel dims:  D={D}, nh={nh}, N={N:,}")

### Token embedding

Every token (byte value 0–255) is mapped to a learnable vector of size D=256.  
Think of it as a lookup table: integer in → float vector out.

After embedding we `unsqueeze(1)` to add a **head dimension**.  
BDH processes data in shape `[B, nh, T, D]` during attention, but the embedding  
starts as `[B, 1, T, D]` — the `1` will broadcast across all `nh` heads automatically.

In [ ]:
with torch.no_grad():
    emb = m.embed(idx)                     # [1, 5, 256]
    x   = emb.unsqueeze(1)                 # [1, 1, 5, 256]
    x   = m.ln(x)                          # normalize — keeps values well-scaled

print(f"embed(idx) shape:  {emb.shape}    ← [B, T, D]")
print(f"after unsqueeze:   {x.shape}  ← [B, 1, T, D]   (1 = head placeholder)")
print()
print(f"'h' embedding (first 8 values): {emb[0, 0, :8].tolist()}")
print(f"'e' embedding (first 8 values): {emb[0, 1, :8].tolist()}")
print("(different characters → different vectors)")

### Layer loop — step 1: Expand to neuron space

Each token starts as a vector of **D=256 numbers** (the compressed internal representation).  
We multiply by `encoder` to get **N=8192 numbers per head** — 32,768 total across 4 heads.

```
x:        [B=1,  1,  T=5, D=256]
encoder:  [nh=4, D=256,   N=8192]
result:   [B=1, nh=4, T=5, N=8192]   ← broadcast over the head dimension
```

**Why expand to such a huge space?**

Think of it as giving each concept its own dedicated neuron. In D=256 dimensions, different concepts have to share space and interfere with each other. In N=8192 dimensions, there is room for separate neurons for "verb", "past tense", "this specific word", "this grammatical role", etc.

After training, each neuron becomes **monosemantic** — it fires for one specific concept and nothing else. This makes the model interpretable: you can look at which neurons fired and know exactly what concept the model is processing.

**What "hello" looks like after expansion (head 0):**
```
'h'  →  [0.0, 0.0, 0.4, 0.0, 0.7, 0.0, 0.3, 0.0, ...]   neurons 2, 4, 7 fire
'e'  →  [0.0, 0.3, 0.0, 0.0, 0.0, 0.5, 0.0, 0.0, ...]   neurons 1, 5 fire
'l'  →  [0.6, 0.0, 0.0, 0.2, 0.0, 0.0, 0.0, 0.4, ...]   neurons 0, 3, 7 fire
'l'  →  [0.6, 0.0, 0.0, 0.2, 0.0, 0.0, 0.0, 0.4, ...]   identical (same byte!)
'o'  →  [0.0, 0.0, 0.2, 0.0, 0.0, 0.3, 0.0, 0.0, ...]   neurons 2, 5 fire
```

After ReLU zeros out negatives:
- With random weights (now): ~50% of neurons survive  
- After training: ~5% survive — each neuron becomes highly selective

The sparse pattern that remains is the model's **concept fingerprint** for this token. This is called `x_sparse` and it answers the question: *"what is this token?"*

In [ ]:
with torch.no_grad():
    # x:       [B=1, 1,  T=5, D=256]
    # encoder: [nh=4, D=256, N=8192]
    # result:  [B=1, nh=4, T=5, N=8192]  ← the "1" in x broadcasts to all 4 heads
    x_latent = x @ m.encoder
    x_sparse = F.relu(x_latent)

print(f"encoder shape:   {m.encoder.shape}  ← [nh, D, N]")
print(f"x_latent shape:  {x_latent.shape}  ← [B, nh, T, N]")
print(f"x_sparse shape:  {x_sparse.shape}  ← same, but negatives zeroed")
print()

# How sparse is it with random (untrained) weights?
active_pct = (x_sparse > 0).float().mean().item()
print(f"Fraction of neurons active: {active_pct:.1%}")
print("(random weights → ~50% active; after training → ~5%)")
print()

# What does a neuron row look like?
token_0_head_0 = x_sparse[0, 0, 0, :]     # all neurons for 'h', head 0
print(f"Neuron activations for 'h' in head 0 — first 20:")
print([f"{v:.2f}" for v in token_0_head_0[:20].tolist()])
print(f"  ({(token_0_head_0 > 0).sum().item()} of {N} neurons are active)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
fig.patch.set_facecolor('#FAFAFA')
fig.suptitle('What "Sparse Activations" Actually Looks Like  (untrained weights → ~50% active; trained → ~5%)',
             fontsize=12, fontweight='bold', y=1.02)

# Use x_latent and x_sparse from the cell above (head 0)
before = x_latent[0, 0].detach().numpy()   # [T=5, N=8192]
after  = x_sparse[0, 0].detach().numpy()   # same, but negatives zeroed

# ── Plot 1: Distribution before ReLU ──────────────────────────────────────────
ax = axes[0]; ax.set_facecolor('#F5F5F5')
ax.hist(before.flatten(), bins=80, color='#1976D2', alpha=0.85, edgecolor='white', lw=0.5)
ax.axvline(0, color='#D32F2F', lw=2.5, ls='--', label='ReLU cutoff')
ax.set_title('Before ReLU\n(all neuron activations)', fontsize=11, fontweight='bold')
ax.set_xlabel('Activation value', fontsize=10); ax.set_ylabel('Count', fontsize=10)
ax.legend(fontsize=9)
ax.text(0.97, 0.95, 'roughly symmetric\naround zero', transform=ax.transAxes,
        ha='right', va='top', fontsize=9, color='#555', style='italic')

# ── Plot 2: After ReLU ─────────────────────────────────────────────────────────
ax = axes[1]; ax.set_facecolor('#F5F5F5')
frac_active = (after > 0).mean()
nonzero = after[after > 0]
ax.hist(nonzero, bins=60, color='#2E7D32', alpha=0.85, edgecolor='white', lw=0.5)
ax.set_title(f'After ReLU — {frac_active*100:.1f}% of neurons survive\n(the rest are exactly zero)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Activation value (>0 only)', fontsize=10); ax.set_ylabel('Count', fontsize=10)
ax.text(0.97, 0.95, f'{(1-frac_active)*100:.1f}%\nzeroed out!',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=10, color='#D32F2F', fontweight='bold', style='italic')

# ── Plot 3: Heatmap — which neurons fire per token ────────────────────────────
ax = axes[2]
show = (after[:, :300] > 0).astype(float)   # first 300 neurons
im = ax.imshow(show, aspect='auto', cmap='Blues', interpolation='nearest', vmin=0, vmax=1)
ax.set_yticks(range(T)); ax.set_yticklabels(list(text), fontsize=12)
ax.set_xlabel('Neuron index (first 300 of 8,192)', fontsize=10)
ax.set_title('Which neurons fire for each character?\n(blue = active, white = zero)',
             fontsize=11, fontweight='bold')
for i in range(T):
    n_on = int((after[i] > 0).sum())
    ax.text(302, i, f'{n_on} active', va='center', fontsize=8, color='#2E7D32')
plt.colorbar(im, ax=ax, shrink=0.8, label='active')

plt.tight_layout()
plt.show()

print(f"\nKey insight: only {frac_active*100:.1f}% of the {N:,} neurons fire per token (untrained).")
print("After training this drops to ~5%. Each neuron becomes specialized — called 'monosemanticity'.")

### Why does sparsity drop to ~5% after training?

It's not explicitly forced — it **emerges** from the architecture. Three pressures combine:

---

**Pressure 1 — The Hebbian gate creates selection pressure**

For a neuron to contribute to the output it must fire in **both** x_sparse and y_sparse:
```
gate = x_sparse × y_sparse
```
If neuron 4 fires randomly for everything, it contributes noise to the gate — it doesn't help predict the next token. Backprop penalises this. So neurons that fire too broadly get pushed to fire less — their encoder weights sharpen until they only fire for specific patterns.

---

**Pressure 2 — The compression step rewards specialization**

After the gate:
```
output = gate @ decoder    [nh*N → D]
```
If 50% of neurons fire, you have ~4000 signals competing to write into D=256 dimensions — they cancel each other out and produce noise.  
If only 5% fire, you have ~400 signals writing into 256 dimensions — far less interference. Each neuron's contribution actually matters.

So the model learns: **fire rarely, fire meaningfully**.

---

**Pressure 3 — ρ retrieval quality demands clean neurons**

If neuron 4 fires for both 'h' and 'e':
```
ρ[4] = 0.7×v['h'] + 0.5×v['e']    ← blurry, mixed signal
```
Querying with 'h' retrieves a blend of 'h' and 'e' content — not useful, higher loss.

But if neuron 4 fires **only** for 'h':
```
ρ[4] = 0.7×v['h']    ← clean, precise signal
```
Querying with 'h' retrieves exactly 'h' content. Lower loss, so backprop reinforces this.

---

**Why ~5% and not 1% or 0.1%?**

Being too sparse means not enough signal gets through → high loss.  
Being too dense means neurons interfere → high loss.  
~5% is where the model settles — the sweet spot where neurons are specialized enough to be useful but numerous enough to be expressive.

The large N (8192) is what makes 5% affordable — you still get ~400 active neurons per token, which is plenty of signal.

### Layer loop — step 2: Linear attention

Each token uses its sparse pattern to **look at past tokens** and ask:  
*"Which past tokens are similar to me, and what were they representing?"*

**Q = K = x_sparse — the same tensor used for both:**
```
Q = rotate(x_sparse, position)    ← "what am I looking for?"
K = rotate(x_sparse, position)    ← "what do I represent?"
V = x                             ← "what content do I carry?" (original D=256, not expanded)
```

No separate projection matrices for Q and K — x_sparse plays both roles. This saves parameters and makes the Hebbian interpretation natural.

**Why V = x and not x_sparse?**  
V carries the actual content to be mixed and passed forward. Using the small D=256 representation keeps V compact. x_sparse (N=8192) is only used for *matching* — deciding who attends to whom.

**Scores for "hello" — which tokens attend to which:**
```
scores = (Q @ Kᵀ).tril(diagonal=-1)   ← lower triangle only, excluding self

         'h'    'e'    'l'    'l'    'o'
'h'  [   0      0      0      0      0  ]   ← first token, nothing to attend to
'e'  [  0.3     0      0      0      0  ]   ← attends to 'h'
'l'  [  0.1    0.5     0      0      0  ]   ← attends to 'h' and 'e'
'l'  [  0.1    0.5    0.8     0      0  ]   ← same content as 'l'[2] but RoPE makes scores different!
'o'  [  0.2    0.3    0.1    0.1     0  ]   ← attends to all past tokens
```

**Key: the two 'l' tokens.**  
Both 'l' tokens have identical x_sparse (same byte). Without RoPE, 'l'[3] attending to 'l'[2] would give the same score as 'l'[2] attending to itself — the model can't tell they're at different positions.  
RoPE rotates each vector by its position before the dot product, so 'l'[3] and 'l'[2] produce a *slightly smaller* dot product than 'l'[2] with itself would. Distance is encoded in the rotation angle.

**No softmax — why:**  
Transformer softmax forces all scores to sum to 1 (relative comparison). BDH uses raw dot products — scores can be any value, can be negative, tokens don't compete. This also makes the ρ accumulation mathematically clean at inference.

**Result:**
```
a_star = scores @ V    [T, D]
```
Each token gets a weighted blend of past tokens' D=256 embeddings. This is what the context has been saying.

In [ ]:
with torch.no_grad():
    yKV = m.attn(Q=x_sparse, K=x_sparse, V=x)
    yKV = m.ln(yKV)

print(f"yKV shape: {yKV.shape}  ← [B, nh, T, D]")
print()

# Look at the raw attention score matrix for one head
# scores[t, s] = how much token t attends to token s
Q_demo = x_sparse[0, 0]   # head 0, [T=5, N=8192]
scores = (Q_demo @ Q_demo.T).tril(diagonal=-1)   # [5, 5]

print("Attention scores (head 0) — rows=query token, cols=key token")
print("(diagonal and upper triangle are 0: can't see yourself or the future)")
print()
for i, row in enumerate(scores.tolist()):
    label = f"  '{text[i]}' attends to: "
    vals  = " ".join(f"{v:6.2f}" for v in row)
    print(label + vals)
print()
print("Column headers:  " + "  ".join(f"  '{c}'  " for c in text))

### Layer loop — step 3: The Hebbian gate

This is the most important step — and the one that makes BDH biologically inspired.

**First, expand a_star to neuron space:**
```
y_sparse = ReLU(a_star @ encoder_v)    [B, nh, T, N]
```
This uses a *different* encoder matrix (`encoder_v`) to expand the attention output.  
The result, `y_sparse`, answers the question: *"given what past context said, which neurons fire?"*

---

**What x_sparse and y_sparse actually represent:**

- **x_sparse** = *"what is this token?"*  
  Comes directly from the current token's embedding. Fires for the token's intrinsic properties:  
  `'h'` → neurons for: consonant, word-start-letter, ...  
  `'.'` → neurons for: sentence-end, punctuation, ...

- **y_sparse** = *"what does the context expect?"*  
  Comes from the attention over past tokens. Fires for what the context is pointing toward:  
  after `"the cat sat on the"` → neurons for: surface, location, noun, ...  
  after `"she picked up the"` → neurons for: graspable-object, thing, ...

---

**The gate — element-wise multiply:**
```python
gate = x_sparse * y_sparse    # element-wise, [B, nh, T, N]
```

A neuron in the gate is nonzero **only if it fired in both**:

```
neuron 42 fires in x_sparse  →  current token has property 42
neuron 42 fires in y_sparse  →  context was expecting property 42
                                              ↓
neuron 42 survives           →  this property is contextually relevant right now
```

Everything that is true about the token but irrelevant to context gets silenced.

**"hello" example — processing 'l' at position 3:**
```
x_sparse['l'][3]  fires: neurons for consonant, repeated-letter, middle-of-word, ...
y_sparse[3]       fires: neurons for letter-follows-l, middle-of-word, vowel-likely-next, ...

gate[3]           fires: middle-of-word, letter-follows-l   ← the overlap
                  dies:  consonant, repeated-letter          ← true but context doesn't care
```

---

**Why the gate makes activations even sparser:**

```
x_sparse alone:  ~50% active  (random weights) or ~5% (trained)
y_sparse alone:  ~50% active  (random weights) or ~5% (trained)

gate = x × y:    much sparser — neuron must fire in BOTH
                 if x and y are independent, P(both fire) = P(x fires) × P(y fires)
                 ~5% × ~5% = ~0.25%  (even sparser after training)
```

In practice the gate isn't quite that sparse because x and y are correlated — but the gate is always sparser than either alone. This double-filtering is where the model's deep sparsity comes from.

---

**This is Hebb's rule:**  
*"Neurons that fire together, wire together."*  

When neuron 42 fires in both x_sparse and y_sparse, it means:  
- this token has property 42  
- the context was expecting property 42  
→ the association "property 42 appears here" gets written into ρ  
→ future similar contexts will retrieve this information

The gate isn't just filtering — it's selecting which associations get strengthened in memory.

In [ ]:
with torch.no_grad():
    y_sparse  = F.relu(yKV @ m.encoder_v)         # [B, nh, T, N]
    xy_sparse = x_sparse * y_sparse                # [B, nh, T, N] — Hebbian gate

print(f"x_sparse active:  {(x_sparse > 0).float().mean():.1%}  (current token neurons)")
print(f"y_sparse active:  {(y_sparse > 0).float().mean():.1%}  (context neurons)")
print(f"xy_sparse active: {(xy_sparse > 0).float().mean():.1%}  (neurons active in BOTH)")
print()
print("The gate is the AND of the two — much sparser than either alone.")
print()

# Concrete example: neuron-by-neuron for token 'l' (index 2) in head 0
t, h = 2, 0
nx = (x_sparse[0, h, t] > 0)
ny = (y_sparse[0, h, t] > 0)
nxy = (xy_sparse[0, h, t] > 0)
print(f"For token '{text[t]}' (pos {t}), head {h}:")
print(f"  x_sparse  has {nx.sum().item():>4} active neurons")
print(f"  y_sparse  has {ny.sum().item():>4} active neurons")
print(f"  xy_sparse has {nxy.sum().item():>4} active neurons  ← only the intersection")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 4, figsize=(15, 4.5))
fig.patch.set_facecolor('#FAFAFA')
fig.suptitle('The Hebbian Gate: only neurons active in BOTH survive',
             fontsize=13, fontweight='bold', y=1.02)

n_show = 300   # show first 300 neurons
h_show = 0     # head 0

xs_np  = (x_sparse[0, h_show, :, :n_show] > 0).float().detach().numpy()
ys_np  = (y_sparse[0, h_show, :, :n_show] > 0).float().detach().numpy()
xys_np = (xy_sparse[0, h_show, :, :n_show] > 0).float().detach().numpy()

data   = [xs_np, ys_np, None, xys_np]
titles = ['x_sparse\n(current token activations)', 'y_sparse\n(attention: context says)',
          '', 'gate = x × y\n(intersection only)']
colors = ['#1565C0', '#6A1B9A', 'white', '#2E7D32']

for i, (ax, d, t, c) in enumerate(zip(axes, data, titles, colors)):
    ax.set_facecolor('#FAFAFA')
    if d is None:
        ax.text(0.5, 0.5, '×', ha='center', va='center', fontsize=60,
                color='#888', transform=ax.transAxes)
        ax.text(0.5, 0.2, 'element-wise\nmultiply', ha='center', va='bottom',
                fontsize=9, color='#888', transform=ax.transAxes, style='italic')
        ax.axis('off')
        continue
    ax.imshow(d, aspect='auto', cmap='Blues', vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(t, fontsize=11, fontweight='bold', color=c)
    if i == 0:
        ax.set_yticks(range(T)); ax.set_yticklabels(list(text), fontsize=12)
    else:
        ax.set_yticks([])
    frac = d.mean()
    ax.set_xlabel(f'{frac*100:.1f}% active', fontsize=9, color=c, fontweight='bold')

axes[3].set_xlabel(f"{xys_np.mean()*100:.1f}% active  ← even sparser!", fontsize=9,
                   color='#2E7D32', fontweight='bold')

fig.text(0.5, -0.06,
         "Hebb's rule: \"neurons that fire together, wire together\"\n"
         "The gate keeps only neurons that agree between current token AND past context — everything else is silenced.",
         ha='center', fontsize=10, color='#444', style='italic')

plt.tight_layout()
plt.show()

print("\nSparsity at each stage (head 0, first 300 neurons):")
print(f"  x_sparse:   {xs_np.mean()*100:.1f}% active  (current token)")
print(f"  y_sparse:   {ys_np.mean()*100:.1f}% active  (from context)")
print(f"  gate x×y:  {xys_np.mean()*100:.1f}% active  (intersection — much sparser!)")

### Layer loop — step 4: Compress back to D and residual

`xy_sparse` is `[B, nh, T, N]` — a huge sparse tensor.  
We need to get back to `[B, 1, T, D]` to feed into the next layer.

Two shape operations happen:
1. `.transpose(1,2).reshape(B, 1, T, N*nh)` — merge all heads into one big flat vector
2. `@ decoder` — project from `nh*N` = 32768 dimensions back down to D=256

Then a **residual connection**: `x = x + y`. This adds the layer's output *on top of* its  
input rather than replacing it. Residuals are why deep networks can be trained at all —  
gradients flow directly back through the `+` without going through the whole layer.

In [ ]:
with torch.no_grad():
    # Merge the head and neuron dimensions, then compress
    flat   = xy_sparse.transpose(1, 2).reshape(B, 1, T, N * nh)
    yMLP   = flat @ m.decoder           # [B, 1, T, D]
    y      = m.ln(yMLP)
    x_new  = m.ln(x + y)               # residual: add layer output to input

print(f"xy_sparse shape:  {xy_sparse.shape}   ← [B, nh, T, N]")
print(f"after transpose:  {xy_sparse.transpose(1,2).shape}   ← [B, T, nh, N]")
print(f"after reshape:    {flat.shape}   ← [B, 1, T, nh*N]  (heads merged)")
print(f"after @ decoder:  {yMLP.shape}  ← [B, 1, T, D]  (compressed back)")
print(f"after residual:   {x_new.shape}  ← same as input, ready for next layer")
print()
print(f"decoder shape: {m.decoder.shape}  ← [nh*N, D]  =  [{N*nh}, {D}]")
print()

# Show that the residual is a gentle update, not a replacement
delta = (x_new - x).abs().mean().item()
scale = x.abs().mean().item()
print(f"Mean |x|:         {scale:.4f}")
print(f"Mean |x_new - x|: {delta:.4f}  ← layer added a small correction on top")

### Why the same weights are reused across all layers — and how the model learns efficiently

**Weight sharing:**  
BDH uses the same `encoder`, `encoder_v`, `decoder` for every layer — no separate weights per layer. This is the Universal Transformer idea: instead of having L different transformations, apply one transformation L times and let the representation refine iteratively.

The model learns a single update rule that, when applied repeatedly, converges to a good answer. Each layer pass refines the representation — early layers build basic structure, later passes refine detail.

---

**How the model learns to use neurons efficiently over time:**

At the start of training, weights are random and ~50% of neurons fire for everything. The model is noisy and makes bad predictions.

Three pressures push neurons toward specialization during training:

**1. The gate creates selection pressure**  
For a neuron to matter, it must fire in both x_sparse AND y_sparse. If neuron 4 fires for everything, it contributes noise to the gate. Backprop penalises noise → neuron 4's weights sharpen until it only fires for specific patterns.

**2. The compression step rewards specificity**  
Gate outputs are compressed back to D=256. If 50% of neurons fire, 4,096 signals compete to write into 256 dimensions — they cancel each other out. If only 5% fire, 400 signals write into 256 dimensions — far less interference, each signal actually matters. The model learns: fire rarely, fire meaningfully.

**3. ρ retrieval quality demands clean neurons**  
If neuron 4 fires for both 'h' and 'e':
```
ρ[4] = 0.7×v['h'] + 0.5×v['e']    ← blurry mixed signal
```
Querying with 'h' retrieves a blend — not useful, higher loss. But if neuron 4 fires only for 'h':
```
ρ[4] = 0.7×v['h']    ← clean signal
```
Lower loss, so backprop reinforces this specialization.

---

**The result: monosemanticity**

After training, each neuron fires for one specific concept. You can inspect which neurons fire for a given input and read off what the model is "thinking about". This is called **monosemanticity** and it's an inherent property of BDH — not something you have to engineer separately.

```
neuron 42:   fires for "past tense verb"
neuron 107:  fires for "proper noun, person name"  
neuron 891:  fires for "end of clause"
...
```

This is fundamentally different from dense models where every neuron participates in every computation and individual neurons are uninterpretable.

---

**Sparsity settles at ~5% — why not lower?**

Too sparse → not enough signal gets through → high loss  
Too dense → neurons interfere with each other → high loss  
~5% → sweet spot where neurons are specialized enough to be useful but numerous enough to carry sufficient information

The large N (8192 per head) is what makes 5% affordable — you still get ~400 active neurons per token. That's plenty of signal for precise computation.

In [ ]:
with torch.no_grad():
    # Run 6 layers manually and watch x evolve
    x = m.ln(m.embed(idx).unsqueeze(1))
    print("How x changes across layers (mean absolute value per layer):")
    print(f"  input:   {x.abs().mean().item():.4f}")

    for layer_i in range(C.n_layer):
        x_s  = F.relu(x @ m.encoder)
        yKV  = m.ln(m.attn(Q=x_s, K=x_s, V=x))
        y_s  = F.relu(yKV @ m.encoder_v)
        xy   = x_s * y_s
        yMLP = xy.transpose(1,2).reshape(B, 1, T, N*nh) @ m.decoder
        x    = m.ln(x + m.ln(yMLP))
        print(f"  layer {layer_i}:  {x.abs().mean().item():.4f}")

print()
print("Same weights, but x is different each time → different computation each layer.")

### Final output: logits and loss

After all layers, `x` is `[B, 1, T, D]`. We reshape to `[B, T, D]` and multiply by  
`lm_head` to get `[B, T, 256]` — one score per possible next byte.

**Cross-entropy loss** measures how surprised the model was by the actual next token.  
- Perfect prediction → loss near 0  
- Random guessing over 256 bytes → loss ≈ log(256) ≈ 5.55  
- A well-trained model on English text → loss ≈ 1.5–2.0

In [ ]:
with torch.no_grad():
    logits, loss = m(idx)

print(f"lm_head shape: {m.lm_head.shape}  ← [D, vocab_size]")
print(f"logits shape:  {logits.shape}   ← [B, T, 256]")
print()
print(f"Random baseline loss: {math.log(256):.3f}")
print(f"Untrained model loss: {loss.item():.3f}")
print()

# For position 0 ('h'), what byte does the untrained model predict comes next?
token_0_logits = logits[0, 0]                          # scores for all 256 bytes
top5_vals, top5_idx = torch.topk(token_0_logits, 5)
print(f"After 'h', the untrained model's top 5 guesses for next byte:")
for val, idx_byte in zip(top5_vals.tolist(), top5_idx.tolist()):
    char = chr(idx_byte) if 32 <= idx_byte < 127 else f"\\x{idx_byte:02x}"
    print(f"  '{char}'  (byte {idx_byte:3d})  score: {val:.3f}")
print("(random, because not trained yet)")

### From logits to the next byte — how generation actually works

After the final layer, each token has a vector of **256 raw scores** (logits) — one per possible next byte.

**Step 1 — Logits: raw unnormalised scores**
```
logits['o'] = [2.1, -0.3, 0.7, ..., 1.4]   ← 256 numbers, any positive or negative value
```
Higher score = model thinks that byte is more likely to come next.

**Step 2 — Softmax: turn scores into probabilities**
```
probs = softmax(logits)   →   [0.04, 0.01, 0.02, ..., 0.12]
```
- All values now between 0 and 1  
- All 256 values sum to exactly 1  
- A large logit gap → very confident prediction

**Step 3 — Temperature: control randomness**
```
logits = logits / temperature

temperature = 1.0  →  sample normally (default)
temperature < 1.0  →  sharper — model is more decisive, more repetitive
temperature > 1.0  →  flatter — model is more random, more creative
```

**Step 4 — Sample: pick the next byte**

You don't always pick the highest probability. You sample randomly, weighted by the probabilities. This is why the model doesn't always say the same thing.

**Step 5 — Append and repeat (autoregressive)**
```
"hell"   →  logits  →  sample 'o'  →  "hello"
"hello"  →  logits  →  sample ' '  →  "hello "
"hello " →  logits  →  sample 'w'  →  "hello w"
...
```
Each new byte is appended to the input and the whole forward pass runs again.  
The model is literally eating its own output — hence "autoregressive".

**Training vs inference — the only difference:**
```
training:   you have the real next byte → compute loss → backprop → update weights
inference:  no real next byte → sample from distribution → append → keep going
```
The forward pass is identical. Only what you do with the logits differs.

---
## Part 2c — The memory state ρ and inference-time learning

This is the part that makes BDH genuinely different from a Transformer at inference time.

### The associativity trick — why ρ works

During training we compute:
```
a_star = (Q @ Kᵀ) @ V
```

Matrix multiplication is **associative**, so you can put the brackets anywhere:
```
a_star = Q @ (Kᵀ @ V)
              └──────┘
                 ρ   ← shape [N, D], fixed size
```

These two are **mathematically identical**. But the second form is special:

- `Kᵀ @ V` is a sum of outer products — one per past token
- You can build it up **incrementally**, one token at a time
- It never grows — always `[N, D]` no matter how long the sequence

```
ρ = 0
for each new token t:
    ρ += outer(k[t], v[t])    ← update: add this token's contribution
    a_star[t] = q[t] @ ρ      ← query: what does past context say?
```

This replaces the entire KV cache with a single fixed matrix.

In [ ]:
# Prove that ρ accumulation gives the exact same result as full attention
# Using the "hello" example from above (m, idx, x, x_sparse, T, D, text all still defined)

with torch.no_grad():
    # ── Mode 1: full attention matrix (training style) ────────────────────────
    full_a_star = m.attn(Q=x_sparse, K=x_sparse, V=x)  # [B=1, nh=4, T=5, D=256]

    # ── Mode 2: ρ accumulation (inference style) — head 0 only ───────────────
    N_dim = x_sparse.shape[-1]             # 8192
    rho   = torch.zeros(N_dim, D)          # [N, D] — starts empty

    rho_outputs = []
    for t in range(T):
        q_t = x_sparse[0, 0, t]           # [N] — query for this token

        # QUERY first (token t only sees tokens 0..t-1)
        a_t = q_t @ rho                    # [N] @ [N, D] = [D]
        rho_outputs.append(a_t)

        # UPDATE state with this token (so future tokens can see it)
        k_t = x_sparse[0, 0, t]           # [N]
        v_t = x[0, 0, t]                  # [D]
        rho = rho + torch.outer(k_t, v_t) # [N, D] += outer product

    rho_a_star  = torch.stack(rho_outputs) # [T, D]
    full_result = full_a_star[0, 0]        # [T, D] — head 0

    print("Full attention result (head 0, first 4 values per token):")
    for t, ch in enumerate(text):
        print(f"  '{ch}': {full_result[t, :4].tolist()}")

    print()
    print("ρ accumulation result (head 0, first 4 values per token):")
    for t, ch in enumerate(text):
        print(f"  '{ch}': {rho_a_star[t, :4].tolist()}")

    print()
    match = torch.allclose(rho_a_star, full_result, atol=1e-4)
    print(f"Identical? {match}  (max diff: {(rho_a_star - full_result).abs().max():.2e})")
    print()
    print(f"ρ shape: {rho.shape}  ← always [N={N_dim}, D={D}], never grows!")
    print(f"KV cache equivalent would be: {T} × {N_dim} + {T} × {D} = {T*(N_dim+D):,} numbers")
    print(f"ρ stores just:                {N_dim} × {D}              = {N_dim*D:,} numbers")


### What ρ looks like as it grows

Each token adds an outer product to ρ. Let's watch ρ accumulate token by token.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, T+1, figsize=(15, 3.5))
fig.patch.set_facecolor('#FAFAFA')
fig.suptitle('ρ accumulating token by token  (showing first 64×64 neurons)',
             fontsize=12, fontweight='bold', y=1.02)

with torch.no_grad():
    rho_viz = torch.zeros(N_dim, D)
    axes[0].imshow(rho_viz[:64, :64].numpy(), cmap='RdBu', vmin=-0.05, vmax=0.05,
                   interpolation='nearest')
    axes[0].set_title('ρ = 0\n(start)', fontsize=10)
    axes[0].axis('off')

    for t, ch in enumerate(text):
        k_t = x_sparse[0, 0, t]
        v_t = x[0, 0, t]
        rho_viz = rho_viz + torch.outer(k_t, v_t)

        ax = axes[t+1]
        ax.imshow(rho_viz[:64, :64].numpy(), cmap='RdBu', vmin=-0.05, vmax=0.05,
                  interpolation='nearest')
        ax.set_title(f"after '{ch}'\n(pos {t})", fontsize=10)
        ax.axis('off')

plt.tight_layout()
plt.show()
print("Each new token adds its 'fingerprint' into ρ.")
print("Any future token can then query ρ to retrieve what past tokens were about.")


### Inference-time Hebbian learning — teaching without backprop

Here's the really cool part.

In a Transformer, if you want the model to "know" a new fact, you have two options:
1. **Fine-tune** (expensive — backprop through the whole model)
2. **Put it in the context** (works but fills up the KV cache)

In BDH, there's a third option: **just run the model on the new text**.

Because ρ accumulates naturally as you process tokens, feeding the model new text
**automatically updates ρ** with associations between those tokens. No backprop.
No gradient computation. Just a forward pass.

This is Hebb's rule:
```
"neurons that fire together, wire together"

when tokens A and B appear together:
    k[A] fires, v[B] fires
    ρ += outer(k[A], v[B])     ← the connection A→B is strengthened in ρ
```

Later, when you ask about A, the model retrieves B — because ρ has that association.

```
                train on base corpus          feed new fact at inference
                (backprop, slow)              (just forward pass, fast)
                      ↓                                ↓
             model learns language            ρ learns specific new info
             weights stay fixed              weights stay fixed, only ρ changes
```

The hackathon goal is to test this: feed new facts at inference, then check if
the model can use them to answer questions it couldn't before.

In [ ]:
# Inference-time Hebbian learning demo
# We start with the ρ we built from "hello", then feed in new text
# and show that ρ changes to encode the new associations

def build_rho(text_str, model=m):
    """Build ρ by processing text_str token by token (no backprop)."""
    with torch.no_grad():
        idx_new  = torch.tensor([[ord(c) for c in text_str]])
        emb_new  = model.embed(idx_new)                        # [1, T, D]
        x_new    = model.ln(emb_new.unsqueeze(1))             # [1, 1, T, D]
        xs_new   = F.relu(x_new @ model.encoder)              # [1, nh, T, N]

        T_new = len(text_str)
        N_dim = xs_new.shape[-1]
        rho   = torch.zeros(N_dim, D)

        for t in range(T_new):
            k_t  = xs_new[0, 0, t]
            v_t  = x_new[0, 0, t]
            rho += torch.outer(k_t, v_t)   # Hebbian update
    return rho

def query_rho(query_char, rho, model=m):
    """Query ρ with a single character and return the retrieved D-dim vector."""
    with torch.no_grad():
        idx_q  = torch.tensor([[ord(query_char)]])
        emb_q  = model.embed(idx_q)
        x_q    = model.ln(emb_q.unsqueeze(1))
        xs_q   = F.relu(x_q @ model.encoder)    # [1, nh, 1, N]
        q      = xs_q[0, 0, 0]                  # [N]
        return q @ rho                            # [D] — retrieved memory

# ── Teach ρ two different phrases ────────────────────────────────────────────
rho_A = build_rho("aardvark")
rho_B = build_rho("zebra")
rho_AB = rho_A + rho_B   # both stored in same ρ

# ── Query with 'a' — should be closer to aardvark memory ─────────────────────
retrieved_from_A  = query_rho('a', rho_A)
retrieved_from_B  = query_rho('a', rho_B)
retrieved_from_AB = query_rho('a', rho_AB)

# cosine similarity — are we closer to aardvark or zebra content?
def cos_sim(a, b):
    return (a @ b / (a.norm() * b.norm())).item()

ref_a = query_rho('a', rho_A)
ref_z = query_rho('z', rho_B)

print("Querying ρ with 'a' (first letter of aardvark):")
print(f"  similarity to aardvark memory: {cos_sim(retrieved_from_AB, ref_a):+.4f}")
print(f"  similarity to zebra memory:    {cos_sim(retrieved_from_AB, ref_z):+.4f}")
print()
print("ρ encodes associations. The more a pattern repeats in the fed text,")
print("the stronger that association becomes in ρ — no backprop needed.")
print()
print(f"ρ norm after 'aardvark':      {rho_A.norm():.3f}")
print(f"ρ norm after 'zebra':         {rho_B.norm():.3f}")
print(f"ρ norm after both (additive): {rho_AB.norm():.3f}")
print()
print("This is why BDH can learn new facts at inference — ρ is writable memory.")


---
## Summary — everything in one place

Here's the complete picture of what we built, from raw text to next-byte prediction.

---

### 1. Tokenization — one byte = one token

```
"hello"  →  [104, 101, 108, 108, 111]   (ord() of each character)
```
- No tokenizer needed — every byte 0-255 is a valid token
- For ASCII English: one character = one byte = one token
- Non-ASCII (e.g. emoji 😊) = multiple bytes = multiple tokens

---

### 2. Embedding lookup — index into a table

```
embed.weight: [256 rows × 256 cols]   ← the lookup table (param 1)

"h" (byte 104) → grab row 104 → [0.3, -0.1, 0.7, ...]   (256 numbers)
```
- `nn.Embedding` is just a matrix — looking up a token = picking a row
- Starts random, trained by backprop
- Result: x of shape [T, D=256] — one 256-dim vector per token

---

### 3. Inside each BDH layer (same weights reused n_layer times)

**Step 1 — Expand to neuron space**
```
x_sparse = ReLU(x @ encoder)           encoder: [nh, D=256, N=8192]  (param 2)
→ shape [nh=4, T, N=8192]
→ ~50% zeros (random weights), ~5% zeros (trained)
```

**Step 2 — Compute Q, K, V**
```
Q = RoPE(x_sparse)    ← rotate by position (no new params)
K = RoPE(x_sparse)    ← same tensor as Q, just rotated
V = x                 ← original D=256 embedding, not expanded
```
RoPE makes tokens at different positions produce different dot products
even if their content (x_sparse) is identical.

**Step 3 — Linear attention**
```
scores  = (Q @ Kᵀ).tril(diagonal=-1)   [T, T] — causal, no softmax
a_star  = scores @ V                    [T, D]
```
Equivalent at inference to:  `a_star[t] = q[t] @ ρ`  where `ρ` is updated incrementally.

**Step 4 — Expand and gate (Hebbian)**
```
y_sparse = ReLU(a_star @ encoder_v)    encoder_v: [nh, D, N]  (param 3)
gate     = x_sparse × y_sparse         element-wise — neurons active in BOTH survive
```
Hebb's rule: a neuron fires only if it responded to both the current token AND the context.

**Step 5 — Compress and residual**
```
output = gate.reshape @ decoder        decoder: [nh*N, D]  (param 4)
x_new  = x + output                   residual connection
```

---

### 4. Output

```
logits = x @ lm_head                   lm_head: [D=256, vocab=256]  (param 5)
→ [T, 256] — one score per possible next byte per position
→ softmax → sample → next byte
```

---

### 5. The 5 learnable parameters (total)

| Name | Shape | Role |
|---|---|---|
| `embed.weight` | [256, 256] | byte → D-dim vector |
| `encoder` | [nh, 256, N] | expand for Q and K |
| `encoder_v` | [nh, 256, N] | expand for V gate |
| `decoder` | [nh×N, 256] | compress back to D |
| `lm_head` | [256, 256] | D → next-byte logits |

All layers share the same weights — BDH applies this one transformation n_layer times.

---

### 6. The memory state ρ

At inference, instead of the full [T, T] attention matrix:
```
ρ = 0
for each token t:
    a_star[t] = q[t] @ ρ              ← query (what does past say?)
    ρ += outer(k[t], v[t])            ← update (record this token)
```
- `ρ` shape: `[N, D]` — fixed forever, never grows
- Replaces the Transformer's KV cache entirely
- Can be updated at inference with new facts (no backprop) — Hebbian learning


---
## Part 3 — Training on tiny Shakespeare

Settings tuned for a T4 GPU (16GB VRAM). Runs ~3000 steps in a few minutes.

In [ ]:
import os, numpy as np, requests
from contextlib import nullcontext

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Mixed precision — bfloat16 is fast and stable on T4/A100; float16 needs a GradScaler
dtype = "bfloat16" if (device.type == "cuda" and torch.cuda.is_bf16_supported()) else "float16" if device.type == "cuda" else "float32"
ptdtype = {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}[dtype]
ctx    = torch.amp.autocast(device_type="cuda", dtype=ptdtype) if device.type == "cuda" else nullcontext()
scaler = torch.amp.GradScaler(device=device.type, enabled=(dtype == "float16"))
print(f"dtype: {dtype}")

# ── Hyperparameters (tuned for T4) ────────────────────────────────────────────
BLOCK_SIZE  = 512   # sequence length
BATCH_SIZE  = 64    # T4 has 16GB — can handle 64 easily
MAX_ITERS   = 3000
LR          = 1e-3
WEIGHT_DECAY = 0.1
LOG_FREQ    = 100

In [ ]:
input_file_path = "input.txt"

def fetch_data():
    if not os.path.exists(input_file_path):
        print("Downloading tiny Shakespeare...")
        url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
        with open(input_file_path, "w") as f:
            f.write(requests.get(url).text)
    print(f"Data ready: {os.path.getsize(input_file_path):,} bytes")

def get_batch(split):
    data = np.memmap(input_file_path, dtype=np.uint8, mode="r")
    data = data[:int(0.9 * len(data))] if split == "train" else data[int(0.9 * len(data)):]
    ix   = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([torch.from_numpy(data[i  :i+BLOCK_SIZE  ].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+BLOCK_SIZE+1].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

fetch_data()

In [ ]:
torch.manual_seed(1337)
model = BDH(BDHConfig()).to(device)
model = torch.compile(model)   # ~2x speedup on CUDA (first step is slow while it compiles)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

x, y = get_batch("train")
loss_acc, loss_steps = 0.0, 0

for step in range(MAX_ITERS):
    with ctx:
        logits, loss = model(x, y)

    x, y = get_batch("train")   # prefetch next batch while GPU does backward

    loss_acc   += loss.item()
    loss_steps += 1

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad()

    if step % LOG_FREQ == 0:
        print(f"step {step:>4}  loss {loss_acc / loss_steps:.3f}")
        loss_acc, loss_steps = 0.0, 0

print("\nDone!")

### Generate text & check sparsity

In [ ]:
model.eval()

# ── Generate ──────────────────────────────────────────────────────────────────
prompt = torch.tensor([list(b"To be or ")], dtype=torch.long, device=device)
with torch.no_grad():
    out = model.generate(prompt, max_new_tokens=200, top_k=3)
print(bytes(out[0].cpu().tolist()).decode(errors="backslashreplace"))

# ── Sparsity ─────────────────────────────────────────────────────────────────
print("\nFraction of neurons active per layer (expect ~5% after training):")
with torch.no_grad():
    C  = model.config
    D, nh = C.n_embd, C.n_head
    N  = D * C.mlp_internal_dim_multiplier // nh
    sample = torch.randint(0, 256, (1, 64), device=device)
    x = model.ln(model.embed(sample).unsqueeze(1))
    for i in range(C.n_layer):
        active = (F.relu(x @ model.encoder) > 0).float().mean().item()
        print(f"  layer {i}: {active:.1%}")
        x_s = F.relu(x @ model.encoder)
        yKV = model.ln(model.attn(Q=x_s, K=x_s, V=x))
        xy  = x_s * F.relu(yKV @ model.encoder_v)
        yMLP = xy.transpose(1,2).reshape(1,1,64,N*nh) @ model.decoder
        x = model.ln(x + model.ln(yMLP))